In [13]:
# Instalando o experta e ajustando a dependência do frozendict
!pip install experta
!pip install frozendict==1.2

In [14]:
# Imports
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping

from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, P, AND, OR, NOT

In [15]:
# --- MODELAGEM DO DOMÍNIO: LIBRAS ---

class ParametroSinal(Fact):
    """
    Recebe os dados físicos do sinal bruto.
    Campos esperados: mao (configuração), local (ponto de articulação), movimento.
    """
    pass

class Expressao(Fact):
    """
    Recebe a expressão do rosto ou corporal.
    Campos esperados: tipo (exemplos: neutra, interrogativa, dor).
    """
    pass

class Contexto(Fact):
    """
    Fato intermediário (Nível 2). Gerado após interpretar os parâmetros físicos.
    Campos esperados: categoria (exemplos: tempo, sentimento, cognitivo).
    """
    pass

class SignificadoFinal(Fact):
    """
    A saída final do sistema (Nível 3).
    Campos esperados: traducao (a palavra ou frase em português).
    """
    pass

In [16]:
# --- TESTANDO A MODELAGEM ---
sinal_teste = ParametroSinal(mao="em_C", local="testa", movimento="nenhum")
print("Validando a criação do Fato:")
print(f"Acessando o local de articulação: {sinal_teste['local']}")

Validando a criação do Fato:
Acessando o local de articulação: testa


In [17]:
class Intencao(Fact):
    pass


class ClassificadorLibras(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        # Lista para guardar o trace
        self.trace_decisoes = []

    # REGRAS DE NÍVEL 1: Deduzindo o contexto (parâmetro -> contexto)

    @Rule(ParametroSinal(mao="em_C", local="testa"))
    def contexto_cognitivo(self):
        motivo = "Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        # Regra dispara e joga novo fato na memória de trabalho
        self.declare(Contexto(categoria="cognitivo"))

    @Rule(ParametroSinal(mao="aberta", local="peito"))
    def contexto_sentimento(self):
        motivo = "Regra Nível 1: Mão aberta no peito -> Contexto 'Sentimento/Emoção'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        self.declare(Contexto(categoria="sentimento"))

    @Rule(ParametroSinal(mao="em_L", local="espaco_neutro"))
    def contexto_tempo(self):
        motivo = "Regra Nível 1: Mão em 'L' no espaço neutro -> Contexto 'Tempo/Calendário'"
        self.trace_decisoes.append(motivo)
        print(f"[DISPARO] {motivo}")
        self.declare(Contexto(categoria="tempo"))

    # NÍVEL 2: Contexto + expressão -> intenção
    # São exigidos dois fatos na memória: contexto e expressão
    @Rule(Contexto(categoria="cognitivo"),
          Expressao(tipo="interrogativa"))
    def intencao_duvida_mental(self):
        motivo = "[R3] Contexto cognitivo e expressão interrogativa gera Intenção 'Dúvida Mental'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(Intencao(tipo="duvida_mental"))

    @Rule(Contexto(categoria="sentimento"),
          Expressao(tipo="dor_ou_tristeza"))
    def intencao_sofrimento(self):
        motivo = "[R4] Contexto Sentimento e Expressão Dor gera Intenção 'Sofrimento'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(Intencao(tipo="sofrimento"))

    # NÍVEL 3: Intenção + movimento -> significado final
    @Rule(Intencao(tipo="duvida_mental"),
          ParametroSinal(movimento="balancar_cabeca"))
    def significado_nao_entendi(self):
        motivo = "[R5] Dúvida Mental e Balançar cabeça significa 'Não entendi'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(SignificadoFinal(traducao="Não entendi / Estou confuso"))

    @Rule(Intencao(tipo="sofrimento"),
          ParametroSinal(movimento="apertar_peito"))
    def significado_angustia(self):
        motivo = "[R6] Sofrimento e Apertar o peito significa 'Angústia'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(SignificadoFinal(traducao="Angústia / Muita Tristeza"))

    # Níveis extras e resolução de conflito
    @Rule(ParametroSinal(mao="em_A", local="cabeca"))
    def contexto_saude(self):
        motivo = "[R7] Mão em 'A' na cabeça gera Contexto 'Saúde/Físico'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(Contexto(categoria="saude"))

    @Rule(Contexto(categoria="saude"),
          Expressao(tipo="dor_ou_tristeza"))
    def intencao_relato_dor(self):
        motivo = "[R8] Contexto Saúde e Expressão Dor gera Intenção 'Relato de Dor'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(Intencao(tipo="relato_dor"))

    # Nível 3 com resolução de conflito (salience e not)
    # Regra 9: Regra de prioridade normal = 0
    # Só dispara se a regra específica de enxaqueca não tiver disparado
    @Rule(Intencao(tipo="relato_dor"),
          NOT(SignificadoFinal(traducao="Enxaqueca / Dor latejante")))
    def significado_dor_generica(self):
        motivo = "[R9] Relato de Dor simples significa 'Estou com dor de cabeça'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(SignificadoFinal(traducao="Estou com dor de cabeça"))

    # Regra 10: Regra específica com salience
    # Se o movimento for "pulsante", ela passa na frente da regra genérica.
    @Rule(Intencao(tipo="relato_dor"),
          ParametroSinal(movimento="pulsante"),
          salience=100) # O motor resolve o conflito dando prioridade pra essa regra.
    def significado_enxaqueca(self):
        motivo = "[R10] Relato de Dor e Movimento Pulsante [ALTA PRIORIDADE] -> 'Enxaqueca'"
        self.trace_decisoes.append(motivo)
        print(f"-> {motivo}")
        self.declare(SignificadoFinal(traducao="Enxaqueca / Dor latejante"))

In [18]:
# --- Testando o nível 1 ---
engine = ClassificadorLibras()
engine.reset()  # Limpa a memória de trabalho antes de começar

# Fato inicial injetado: o que a câmera ou luva leu do usuário
engine.declare(ParametroSinal(mao="em_C", local="testa", movimento="nenhum"))

# Inicia o raciocínio
print("Iniciando o raciocínio do motor...\n")
engine.run()

print("\n--- RESUMO ---")
print("Rastro de decisões (Trace):", engine.trace_decisoes)
print("\nFatos que estão na memória final:")
for f_id, f_data in engine.facts.items():
    print(f"ID {f_id}: {f_data}")

Iniciando o raciocínio do motor...

[DISPARO] Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'

--- RESUMO ---
Rastro de decisões (Trace): ["Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'"]

Fatos que estão na memória final:
ID 0: <f-0>
ID 1: <f-1>
ID 2: <f-2>


In [19]:
# Caso de Teste 1
engine = ClassificadorLibras()
engine.reset()

# Fatos iniciais fornecidos ao sistema (simulando a leitura dos sensores)
engine.declare(ParametroSinal(mao="em_C", local="testa", movimento="balancar_cabeca"))
engine.declare(Expressao(tipo="interrogativa"))

print("Iniciando a dedução do sinal...\n")
engine.run()

print("\n--- TRADUÇÃO FINAL ---")
for fato in engine.facts.values():
    if isinstance(fato, SignificadoFinal):
        print(f"Sinal traduzido: {fato['traducao']}")

Iniciando a dedução do sinal...

[DISPARO] Regra Nível 1: Mão em 'C' na testa -> Contexto 'Cognitivo/Mental'
-> [R3] Contexto cognitivo e expressão interrogativa gera Intenção 'Dúvida Mental'
-> [R5] Dúvida Mental e Balançar cabeça significa 'Não entendi'

--- TRADUÇÃO FINAL ---
Sinal traduzido: Não entendi / Estou confuso


In [20]:
# --- Caso de teste 2: Resolução de conflito com salience ---
engine = ClassificadorLibras()
engine.reset()

# Fatos iniciais com movimento pulsante
engine.declare(ParametroSinal(mao="em_A", local="cabeca", movimento="pulsante"))
engine.declare(Expressao(tipo="dor_ou_tristeza"))

print("Iniciando a dedução do Caso 2 (Conflito)...\n")
engine.run()

print("\n--- TRADUÇÃO FINAL ---")
for fato in engine.facts.values():
    if isinstance(fato, SignificadoFinal):
        print(f"Sinal traduzido: {fato['traducao']}")

Iniciando a dedução do Caso 2 (Conflito)...

-> [R7] Mão em 'A' na cabeça gera Contexto 'Saúde/Físico'
-> [R8] Contexto Saúde e Expressão Dor gera Intenção 'Relato de Dor'
-> [R10] Relato de Dor e Movimento Pulsante [ALTA PRIORIDADE] -> 'Enxaqueca'

--- TRADUÇÃO FINAL ---
Sinal traduzido: Enxaqueca / Dor latejante
